In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.animation as animation
import matplotlib.cm as cm
import json
from tqdm import tqdm
from PIL import Image
import gc
import warnings
import sys, importlib
from pathlib import Path
import random

project_root = Path.cwd().parent if Path.cwd().name == "multi_game_data" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import funcs
importlib.reload(funcs)
%matplotlib inline
warnings.filterwarnings('ignore')

# Tomographic Reconstruction of NBA Games - Multi Game Cleaning, Parsing and Visualizations Before Simulations 

Much of this file contains cleaning similar to that in `NBA_Analysis.ipynb`, but is much more streamlined and applied to multiple games. However, we do introduce some domain knowledge to enhance our data cleaning efforts which we apply retroactively to `NBA_Analysis.ipynb`.

## Possession-clock correction and tracking coverage

The possession CSV parser often records an event row with equal raw start and end times. A blanket interpretation of those rows as independent zero-duration possessions loses movement and can mislabel points. The corrected pipeline below first shifts every segment start to the preceding full-game segment end. It then merges only a narrowly defined continuation: the same team, same period and same end time as the preceding segment, with exactly one point scored. These 13 rows are clock-stopped free-throw continuations. Source and original possession identifiers are retained in an explicit source-to-corrected mapping.

Eight other stopped-clock Raptors rows do not meet that rule. They remain distinct in possession metadata and point totals, are explicitly flagged, and are excluded from spatial extraction because no movement interval can be inferred safely. Missing tracking intervals are reported rather than filled with fabricated frames. Repeated tracking timestamps are reduced with `tail(1)`, so one physical moment is retained per corrected possession and timestamp.

This correction intentionally does **not** classify how a possession began. Turnovers, defensive rebounds, and other defensive-start tracking remain out of scope until that analysis is needed. Existing generated CSVs remain products of their prior run until this notebook is rerun from the top.

In [ ]:
from pathlib import Path

path = "*_possessions.csv"
possession_files = sorted(Path(".").glob(path))

# Read only the raw possession csvs; exclude generated analysis csvs like multi_game_data.csv
dfs = [pd.read_csv(file) for file in possession_files]
dfs_dict = {file.stem: pd.read_csv(file) for file in possession_files}

if not dfs_dict:
    raise FileNotFoundError(f"No possession CSVs found with pattern {path!r} in {Path.cwd()}")

In [ ]:
new_cols = [
    "game_id",
    "ORIGINAL_POSSESSION_NUMBER",
    "period",
    "possession_start",
    "possession_end",
    "possession_team",
    "points_scored",
    "event_count",
]

for key, df in dfs_dict.items():
    df = df.copy()
    df.columns = new_cols
    df = df[df["possession_team"].notna() & df["possession_team"].ne(0)].reset_index(drop=True)
    df["possession_team"] = df["possession_team"].astype("int64")
    df["_source_row"] = np.arange(len(df))
    df["possession_start_raw"] = df["possession_start"]

    # A segment begins when the preceding full-game segment ended. This fixes
    # parser rows whose event-level start and end were recorded at one clock value.
    df["possession_start"] = df["possession_end"].shift(fill_value=0)
    previous = df.shift()
    same_team = df["possession_team"].eq(previous["possession_team"])
    same_period = df["period"].eq(previous["period"])
    same_end_time = df["possession_end"].eq(previous["possession_end"])

    # Merge only confirmed one-point, same-team clock-stopped continuations.
    df["is_confirmed_continuation"] = (
        same_team & same_period & same_end_time & df["points_scored"].eq(1)
    )
    df["is_stopped_clock_segment"] = df["possession_start"].eq(df["possession_end"])
    df["is_unmerged_stopped_clock"] = (
        df["is_stopped_clock_segment"] & ~df["is_confirmed_continuation"]
    )
    dfs_dict[key] = df

In [ ]:
# Check scores, all are correct
for key, df in dfs_dict.items():
    score = df.groupby("possession_team")["points_scored"].sum()
    print(key, score)

In [ ]:
# Build corrected Raptors possessions while preserving every source identifier.
RAPTORS_TEAM_ID = 1610612761
raptors_segments_dict = {}
source_to_corrected_records = []

for key, all_team_segments in dfs_dict.items():
    tracking_game_id = key.replace("_possessions", "")
    segments = all_team_segments.loc[
        all_team_segments["possession_team"].eq(RAPTORS_TEAM_ID)
    ].copy()
    segments["tracking_game_id"] = tracking_game_id
    segments["SOURCE_POSSESSION_NUMBER"] = np.arange(1, len(segments) + 1)
    segments["possession_number"] = (
        ~segments["is_confirmed_continuation"]
    ).cumsum().astype(int)

    corrected = (
        segments.groupby("possession_number", as_index=False, sort=True)
        .agg(
            game_id=("game_id", "first"),
            tracking_game_id=("tracking_game_id", "first"),
            period=("period", "first"),
            possession_start=("possession_start", "first"),
            possession_end=("possession_end", "last"),
            possession_team=("possession_team", "first"),
            points_scored=("points_scored", "sum"),
            event_count=("event_count", "sum"),
            source_possession_numbers=(
                "SOURCE_POSSESSION_NUMBER",
                lambda values: tuple(int(value) for value in values),
            ),
            original_possession_numbers=(
                "ORIGINAL_POSSESSION_NUMBER",
                lambda values: tuple(int(value) for value in values),
            ),
            contains_confirmed_continuation=("is_confirmed_continuation", "any"),
            contains_unmerged_stopped_clock=("is_unmerged_stopped_clock", "any"),
        )
    )
    corrected["possession_duration"] = (
        corrected["possession_end"] - corrected["possession_start"]
    )
    corrected["is_spatially_eligible"] = (
        ~corrected["contains_unmerged_stopped_clock"]
        & corrected["possession_duration"].gt(0)
    )
    corrected["scored"] = corrected["points_scored"].gt(0).astype(int)

    # Source intervals drive extraction; corrected labels drive every downstream group.
    segments = segments.rename(
        columns={
            "possession_start": "source_possession_start",
            "possession_end": "source_possession_end",
            "points_scored": "source_points_scored",
        }
    ).merge(
        corrected[[
            "possession_number",
            "possession_start",
            "possession_end",
            "points_scored",
            "scored",
            "contains_confirmed_continuation",
            "contains_unmerged_stopped_clock",
            "is_spatially_eligible",
        ]],
        on="possession_number",
        how="left",
        validate="many_to_one",
    )

    source_to_corrected_records.append(
        segments[[
            "tracking_game_id",
            "ORIGINAL_POSSESSION_NUMBER",
            "SOURCE_POSSESSION_NUMBER",
            "possession_number",
            "is_confirmed_continuation",
            "is_unmerged_stopped_clock",
            "is_spatially_eligible",
        ]]
    )
    raptors_segments_dict[key] = segments
    dfs_dict[key] = corrected

source_to_corrected_possession_map = pd.concat(
    source_to_corrected_records, ignore_index=True
)

source_points = sum(
    int(df["source_points_scored"].sum()) for df in raptors_segments_dict.values()
)
corrected_points = sum(int(df["points_scored"].sum()) for df in dfs_dict.values())
if source_points != corrected_points:
    raise ValueError("Corrected possession aggregation changed the Raptors point total.")
if source_to_corrected_possession_map.duplicated(
    ["tracking_game_id", "SOURCE_POSSESSION_NUMBER"]
).any():
    raise ValueError("A Raptors source segment maps to more than one corrected possession.")

In [ ]:
dfs_dict['0021500009_possessions']
# The single-game notebook uses a different source export for this game (103 Toronto
# source rows versus 102 here). IDs are therefore not interchangeable; use
# source_to_corrected_possession_map rather than matching by row number.

In [ ]:
source_segment_counts = [len(df) for df in raptors_segments_dict.values()]
possesions = [len(df) for df in dfs_dict.values()]  # corrected possessions; retained for later cells
confirmed_continuations = sum(
    int(df["is_confirmed_continuation"].sum()) for df in raptors_segments_dict.values()
)
unmerged_stopped_clock = sum(
    int(df["contains_unmerged_stopped_clock"].sum()) for df in dfs_dict.values()
)

print(f"Raptors source segments: {sum(source_segment_counts)}")
print(f"Confirmed one-point continuations merged: {confirmed_continuations}")
print(f"Corrected Raptors possessions: {sum(possesions)}")
print(f"Distinct stopped-clock possessions retained but excluded spatially: {unmerged_stopped_clock}")
print(f"Average corrected possessions per game: {sum(possesions) / len(possesions):.1f}")

In [ ]:
# Convert cumulative elapsed seconds to the period countdown clock used by tracking.
def elapsed_to_period_clock(df, time_columns):
    out = df.copy()
    for column in time_columns:
        out[column] = out["period"] * 720 - out[column]
    return out

for key in dfs_dict:
    dfs_dict[key] = elapsed_to_period_clock(
        dfs_dict[key], ["possession_start", "possession_end"]
    )
    raptors_segments_dict[key] = elapsed_to_period_clock(
        raptors_segments_dict[key],
        [
            "source_possession_start",
            "source_possession_end",
            "possession_start",
            "possession_end",
        ],
    )

In [ ]:
# proportion of scoring/non-scoring plays
# looks good
scores = []

for key, df in dfs_dict.items():
    scores.append(sum(df["scored"]))

print(sum(scores)/sum(possesions))

In [ ]:
all_ball_data = []

for key, source_segments in raptors_segments_dict.items():
    game_id = key.replace("_possessions", "")
    json_path = f"{game_id}.json"
    spatial_segments = source_segments.loc[
        source_segments["is_spatially_eligible"]
    ].copy()

    print(f"\nProcessing Game {game_id}")
    with open(json_path, "r") as f:
        tracking_data = json.load(f)

    quarters = tracking_data.get("quarters", {})
    home_team_id = tracking_data["home"]["teamid"]
    is_home = int(home_team_id == RAPTORS_TEAM_ID)
    home_away = "home" if is_home else "away"
    ball_pos_data = []

    # Extract each original source interval, but attach its corrected possession
    # number and corrected scoring label to every movement row.
    for _, row in tqdm(
        spatial_segments.iterrows(),
        total=len(spatial_segments),
        desc=f"Game {game_id}",
    ):
        period = str(int(row["period"]))
        source_start = row["source_possession_start"]
        source_end = row["source_possession_end"]
        moments = quarters.get(period, [])
        if not moments:
            continue

        filtered_moments = [
            {
                "game_id": game_id,
                "home_away": home_away,
                "is_home": is_home,
                "period": int(period),
                "time": moment[2],
                "x": moment[5][0][2],
                "y": moment[5][0][3],
                "z": moment[5][0][4],
                "source_possession_start": source_start,
                "source_possession_end": source_end,
                "possession_start": row["possession_start"],
                "possession_end": row["possession_end"],
                "ORIGINAL_POSSESSION_NUMBER": int(row["ORIGINAL_POSSESSION_NUMBER"]),
                "SOURCE_POSSESSION_NUMBER": int(row["SOURCE_POSSESSION_NUMBER"]),
                "possession_number": int(row["possession_number"]),
                "source_points_scored": int(row["source_points_scored"]),
                "points_scored": int(row["points_scored"]),
                "scored": int(row["scored"]),
                "contains_confirmed_continuation": bool(
                    row["contains_confirmed_continuation"]
                ),
            }
            for moment in moments
            if isinstance(moment, list)
            and len(moment) > 5
            and len(moment[5]) > 0
            and source_end <= moment[2] <= source_start
        ]
        ball_pos_data.extend(filtered_moments)

    all_ball_data.append(pd.DataFrame(ball_pos_data))
    del tracking_data, quarters, ball_pos_data
    gc.collect()

master_ball_pos_df = pd.concat(all_ball_data, ignore_index=True)
print(f"\nTotal extracted source-interval moments: {len(master_ball_pos_df)}")
print(master_ball_pos_df.head())
print(master_ball_pos_df.columns)

In [ ]:
master_ball_pos_df

In [ ]:
df = master_ball_pos_df.copy()
df["_order"] = df.index

# Keep exactly one physical tracking row for a repeated clock timestamp.
clean_possessions_df = (
    df.groupby(
        ["game_id", "period", "possession_number", "time"],
        group_keys=False,
        dropna=False,
    )
    .tail(1)
    .sort_values("_order")
    .drop(columns="_order")
    .reset_index(drop=True)
)

# Assemble corrected metadata and validate every observed source-to-corrected link.
corrected_metadata = []
for corrected in dfs_dict.values():
    game_metadata = corrected.copy()
    game_metadata["source_game_id"] = game_metadata["game_id"]
    game_metadata["game_id"] = game_metadata["tracking_game_id"].astype(str)
    corrected_metadata.append(game_metadata)
corrected_possession_metadata_df = pd.concat(corrected_metadata, ignore_index=True)
possession_key = ["game_id", "possession_number"]

if corrected_possession_metadata_df.duplicated(possession_key).any():
    raise ValueError("Corrected possession identifiers are not unique within game.")

observed_source_map = clean_possessions_df[[
    "game_id", "SOURCE_POSSESSION_NUMBER", "possession_number"
]].drop_duplicates()
expected_source_map = source_to_corrected_possession_map[[
    "tracking_game_id", "SOURCE_POSSESSION_NUMBER", "possession_number"
]].rename(
    columns={
        "tracking_game_id": "game_id",
        "possession_number": "expected_possession_number",
    }
)
source_map_check = observed_source_map.merge(
    expected_source_map,
    on=["game_id", "SOURCE_POSSESSION_NUMBER"],
    how="left",
    validate="many_to_one",
)
if source_map_check["expected_possession_number"].isna().any():
    raise ValueError("Movement contains an unknown Raptors source possession.")
if source_map_check["possession_number"].ne(
    source_map_check["expected_possession_number"]
).any():
    raise ValueError("Movement was assigned to the wrong corrected possession.")

movement_labels = (
    clean_possessions_df.groupby(possession_key, as_index=False)
    .agg(
        tracking_rows=("time", "size"),
        unique_tracking_times=("time", "nunique"),
        observed_points_scored=("points_scored", "first"),
        observed_scored=("scored", "first"),
        points_label_count=("points_scored", "nunique"),
        scored_label_count=("scored", "nunique"),
    )
)
unknown_movement_keys = movement_labels.merge(
    corrected_possession_metadata_df[possession_key],
    on=possession_key,
    how="left",
    indicator=True,
).query("_merge == 'left_only'")
if not unknown_movement_keys.empty:
    raise ValueError("Movement contains a corrected possession absent from metadata.")

possession_tracking_coverage_df = corrected_possession_metadata_df.merge(
    movement_labels,
    on=possession_key,
    how="left",
    validate="one_to_one",
)
possession_tracking_coverage_df["is_represented"] = (
    possession_tracking_coverage_df["tracking_rows"].notna()
)
possession_tracking_coverage_df["is_missing_tracking"] = (
    possession_tracking_coverage_df["is_spatially_eligible"]
    & ~possession_tracking_coverage_df["is_represented"]
)
possession_tracking_coverage_df["is_excluded_stopped_clock"] = (
    ~possession_tracking_coverage_df["is_spatially_eligible"]
)

represented = possession_tracking_coverage_df["is_represented"]
if possession_tracking_coverage_df.loc[represented, "points_label_count"].ne(1).any():
    raise ValueError("A corrected possession has conflicting movement point labels.")
if possession_tracking_coverage_df.loc[represented, "scored_label_count"].ne(1).any():
    raise ValueError("A corrected possession has conflicting movement scoring labels.")
if possession_tracking_coverage_df.loc[represented, "points_scored"].ne(
    possession_tracking_coverage_df.loc[represented, "observed_points_scored"]
).any():
    raise ValueError("Movement point labels do not match corrected metadata.")
if possession_tracking_coverage_df.loc[represented, "scored"].ne(
    possession_tracking_coverage_df.loc[represented, "observed_scored"]
).any():
    raise ValueError("Movement scored labels do not match corrected metadata.")
if possession_tracking_coverage_df.loc[
    possession_tracking_coverage_df["is_excluded_stopped_clock"], "is_represented"
].any():
    raise ValueError("A stopped-clock possession was unexpectedly given spatial frames.")

missing_eligible_possessions_df = possession_tracking_coverage_df.loc[
    possession_tracking_coverage_df["is_missing_tracking"]
].copy()
excluded_stopped_clock_possessions_df = possession_tracking_coverage_df.loc[
    possession_tracking_coverage_df["is_excluded_stopped_clock"]
].copy()
movement_coverage_summary_by_game = (
    possession_tracking_coverage_df.groupby("game_id", as_index=False)
    .agg(
        corrected_possessions=("possession_number", "size"),
        spatially_eligible=("is_spatially_eligible", "sum"),
        represented_possessions=("is_represented", "sum"),
        missing_tracking=("is_missing_tracking", "sum"),
        excluded_stopped_clock=("is_excluded_stopped_clock", "sum"),
        metadata_points=("points_scored", "sum"),
    )
)

movement_coverage_summary = pd.Series(
    {
        "Raptors source segments": len(source_to_corrected_possession_map),
        "Confirmed continuations merged": int(
            source_to_corrected_possession_map["is_confirmed_continuation"].sum()
        ),
        "Corrected Raptors possessions": len(possession_tracking_coverage_df),
        "Spatially eligible corrected possessions": int(
            possession_tracking_coverage_df["is_spatially_eligible"].sum()
        ),
        "Represented corrected possessions": int(represented.sum()),
        "Eligible possessions missing tracking": len(missing_eligible_possessions_df),
        "Stopped-clock possessions excluded spatially": len(
            excluded_stopped_clock_possessions_df
        ),
        "Raptors points in corrected metadata": int(
            possession_tracking_coverage_df["points_scored"].sum()
        ),
        "Points in spatially eligible possessions": int(
            possession_tracking_coverage_df.loc[
                possession_tracking_coverage_df["is_spatially_eligible"], "points_scored"
            ].sum()
        ),
        "Points represented in movement": int(
            possession_tracking_coverage_df.loc[represented, "points_scored"].sum()
        ),
        "Clean movement rows": len(clean_possessions_df),
    },
    name="count",
)

display(movement_coverage_summary.to_frame())
display(movement_coverage_summary_by_game)
display(missing_eligible_possessions_df)
display(excluded_stopped_clock_possessions_df)

In [ ]:
clean_possessions_df.describe()

## Discretizing Court and Binning Movement

Introducing logic for home/away games so movement remains consistent, going left to right.

In [ ]:
def heat_invert(df, img, ax, title="Ball Movement Heatmap"):
    df = df.copy()
    court_img = Image.open(img)

    court_length, court_width = 94, 50
    x_bins, y_bins = 20, 10
    mid = (x_bins - 1) / 2  # 9.5 for 20 bins

    # --- bin ---
    df["x_bin"] = np.clip((df["x"] / court_length * x_bins).astype(int), 0, x_bins - 1)
    df["y_bin"] = np.clip((df["y"] / court_width  * y_bins).astype(int), 0, y_bins - 1)

    # Helper: rotate 180° in bin space
    def rotate_180(mask):
        df.loc[mask, "x_bin"] = (x_bins - 1) - df.loc[mask, "x_bin"]
        df.loc[mask, "y_bin"] = (y_bins - 1) - df.loc[mask, "y_bin"]

    # --- Step 1: halftime rule (your best performer) ---
    halftime_flip = (
        ((df["is_home"] == 1) & (df["period"] >= 3)) |
        ((df["is_home"] == 0) & (df["period"] <= 2))
    )
    rotate_180(halftime_flip)

    # --- Step 2: per-game clamp to force LEFT→RIGHT ---
    # After halftime normalization, some games may still be stored reversed.
    # Flip the whole game (rotate 180°) if its x mass is on the wrong side.
    game_mean = df.groupby("game_id")["x_bin"].mean()
    games_still_wrong = game_mean[game_mean < mid].index  # still left-heavy → rotate
    rotate_180(df["game_id"].isin(games_still_wrong))

    # --- heatmap ---
    heatmap = np.zeros((x_bins, y_bins))
    np.add.at(heatmap, (df["x_bin"], df["y_bin"]), 1)
    heatmap_norm = heatmap / heatmap.max() if heatmap.max() > 0 else heatmap

    ax.imshow(court_img, extent=[0, x_bins, 0, y_bins], aspect="auto")
    im = ax.imshow(
        heatmap_norm.T,
        cmap="hot",
        origin="lower",
        extent=[0, x_bins, 0, y_bins],
        alpha=0.6,
    )
    ax.set_title(title)
    return im

In [ ]:
for game in clean_possessions_df["game_id"].unique():
    fig, ax = plt.subplots(figsize=(8,4))
    game_df = clean_possessions_df[clean_possessions_df["game_id"] == game]
    heat_invert(game_df, "court.jpg", ax, title=f"{game} normalized")
    plt.show()

In [ ]:
def bin_and_flip(
    df: pd.DataFrame,
    court_length: float = 94,
    court_width: float = 50,
    x_bins: int = 20,
    y_bins: int = 10,
    game_col: str = "game_id",
    period_col: str = "period",
    is_home_col: str = "is_home",
    x_col: str = "x",
    y_col: str = "y",
) -> pd.DataFrame:
    """
    Adds x_bin, y_bin and applies:
      1) halftime rule rotation (180° in bin space)
      2) per-game clamp rotation to force consistent orientation
    Returns a copy of df with x_bin, y_bin.
    """
    out = df.copy()
    mid = (x_bins - 1) / 2

    # --- bin ---
    out["x_bin"] = np.clip((out[x_col] / court_length * x_bins).astype(int), 0, x_bins - 1)
    out["y_bin"] = np.clip((out[y_col] / court_width  * y_bins).astype(int), 0, y_bins - 1)

    def rotate_180(mask: pd.Series) -> None:
        out.loc[mask, "x_bin"] = (x_bins - 1) - out.loc[mask, "x_bin"]
        out.loc[mask, "y_bin"] = (y_bins - 1) - out.loc[mask, "y_bin"]

    # --- Step 1: halftime rule (NBA basket switch) ---
    halftime_flip = (
        ((out[is_home_col] == 1) & (out[period_col] >= 3)) |
        ((out[is_home_col] == 0) & (out[period_col] <= 2))
    )
    rotate_180(halftime_flip)

    # --- Step 2: per-game clamp (fix games stored reversed) ---
    game_mean = out.groupby(game_col)["x_bin"].mean()
    games_still_wrong = game_mean[game_mean < mid].index
    rotate_180(out[game_col].isin(games_still_wrong))

    return out

def plot_heatmap_bins(
    df_binned: pd.DataFrame,
    img_path: str,
    ax,
    title: str = "Ball Movement Heatmap",
    x_bins: int = 20,
    y_bins: int = 10,
    cmap: str = "hot",
    alpha: float = 0.6,
):
    """
    Plots a heatmap using precomputed x_bin, y_bin (no flipping done here).
    """
    court_img = Image.open(img_path)

    heatmap = np.zeros((x_bins, y_bins))
    np.add.at(heatmap, (df_binned["x_bin"], df_binned["y_bin"]), 1)

    heatmap_norm = heatmap / heatmap.max() if heatmap.max() > 0 else heatmap

    ax.imshow(court_img, extent=[0, x_bins, 0, y_bins], aspect="auto")
    im = ax.imshow(
        heatmap_norm.T,
        cmap=cmap,
        origin="lower",
        extent=[0, x_bins, 0, y_bins],
        alpha=alpha,
    )
    ax.set_title(title)
    ax.set_xlabel("Court Length Bins")
    ax.set_ylabel("Court Width Bins")
    return im

In [ ]:
binned = bin_and_flip(clean_possessions_df)

binned.to_csv("multi_game_data.csv", index=False)

In [ ]:
project_root = Path.cwd().parent if Path.cwd().name == "multi_game_data" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

binned = funcs.bin_and_flip(clean_possessions_df)
funcs.plot_scoring_movement_heatmaps_by_game(
    binned,
    img_path="court.jpg",
    title="Multi-Game Ball Movement Heatmaps by Game and Play Outcome",
)

plt.show()



In [ ]:
# Plot raw possession paths for each game, split by quarter and play outcome.
# Blue squares mark possession starts; red X markers mark possession ends.
for game_id, game_df in clean_possessions_df.groupby("game_id", sort=True):
    periods = sorted(game_df["period"].dropna().unique())

    funcs.plot_raw_movements_by_game_period(
        game_df,
        img_path="court.jpg",
        title=f"Game {game_id}: Raw Ball Movement by Quarter and Play Outcome",
        figsize=(18, max(4, 3 * len(periods))),
        line_alpha=0.55,
        line_width=1.2,
        marker_size=18,
    )


## Possession level cleaning

Methods to eliminate oddities in data. Removing possessions that cross over half twice and analyzing any possessions that are longer than the expected maximum length (24 seconds) of a given possession. Obviously some possessions will be above this if they reset the shot clock, but visualizing these possessions will help us understand if anything is tracked incorrectly or will present issues when modeling. What we do find here is that some possessions cross over the half court line twice or more, which is a foul during the game. Because we cannot easily diagnose the errors occuring here, we will simply drop them for now. These plays and the long plays will isolated and visualized below before removing.

In [ ]:
long_possessions = []

for key, df in dfs_dict.items():
    game_long_possessions = df.copy()
    game_long_possessions["possession_duration"] = (
        game_long_possessions["possession_end"] - game_long_possessions["possession_start"]
    ).abs()
    game_long_possessions = game_long_possessions[game_long_possessions["possession_duration"] > 24]
    game_long_possessions.insert(0, "source_file", key)
    long_possessions.append(game_long_possessions)

long_possessions_df = (
    pd.concat(long_possessions, ignore_index=True)
    if long_possessions
    else pd.DataFrame()
)
long_possessions_df["tracking_game_id"] = long_possessions_df["source_file"].str.replace(
    "_possessions", "", regex=False
)

long_possessions_df = long_possessions_df.sort_values(
    ["tracking_game_id", "period", "possession_number"]
).reset_index(drop=True)

print(f"Possessions longer than 24 seconds: {len(long_possessions_df)}")
with pd.option_context("display.max_rows", None):
    display(long_possessions_df)

In [ ]:
# Tracking rows for the long possessions, used for movement inspection and plotting
if "tracking_game_id" not in long_possessions_df.columns:
    long_possessions_df = long_possessions_df.copy()
    if "source_file" in long_possessions_df.columns:
        long_possessions_df["tracking_game_id"] = long_possessions_df["source_file"].str.replace(
            "_possessions", "", regex=False
        )
    else:
        long_possessions_df["tracking_game_id"] = long_possessions_df["game_id"].astype(str).str.zfill(10)

long_possession_keys = long_possessions_df[
    ["tracking_game_id", "period", "possession_number"]
].drop_duplicates()

long_possession_movements_df = clean_possessions_df.merge(
    long_possession_keys,
    left_on=["game_id", "period", "possession_number"],
    right_on=["tracking_game_id", "period", "possession_number"],
    how="inner",
).drop(columns="tracking_game_id")

long_possession_movements_df = long_possession_movements_df.sort_values(
    ["game_id", "period", "possession_number", "time"],
    ascending=[True, True, True, False],
).reset_index(drop=True)

def count_half_court_crossings(possession_df, half_court_x=47):
    possession_df = possession_df.sort_values("time", ascending=False)
    side = np.sign(possession_df["x"] - half_court_x).replace(0, np.nan).ffill().bfill()
    if side.empty or side.isna().all():
        return 0
    return int(side.ne(side.shift()).sum() - 1)

all_half_court_crossings_df = (
    clean_possessions_df.groupby(["game_id", "period", "possession_number"])
    .apply(count_half_court_crossings)
    .rename("half_court_crossings")
    .reset_index()
)

all_possessions_with_crossings_df = clean_possessions_df[
    ["game_id", "period", "possession_number", "possession_start", "possession_end", "scored"]
].drop_duplicates().merge(
    all_half_court_crossings_df,
    on=["game_id", "period", "possession_number"],
    how="left",
)
all_possessions_with_crossings_df["half_court_crossings"] = (
    all_possessions_with_crossings_df["half_court_crossings"].fillna(0).astype(int)
)

dropped_half_court_crossers_df = all_possessions_with_crossings_df[
    all_possessions_with_crossings_df["half_court_crossings"] >= 2
].copy()

long_possessions_filtered_df = long_possessions_df.merge(
    all_half_court_crossings_df,
    left_on=["tracking_game_id", "period", "possession_number"],
    right_on=["game_id", "period", "possession_number"],
    how="left",
    suffixes=("", "_tracking"),
)
long_possessions_filtered_df["half_court_crossings"] = (
    long_possessions_filtered_df["half_court_crossings"].fillna(0).astype(int)
)
long_possessions_filtered_df = long_possessions_filtered_df.drop(
    columns=[col for col in ["game_id_tracking"] if col in long_possessions_filtered_df.columns]
)

long_possessions_filtered_df = long_possessions_filtered_df[
    long_possessions_filtered_df["half_court_crossings"] < 2
].copy()

long_possession_movements_df = long_possession_movements_df.merge(
    all_half_court_crossings_df,
    on=["game_id", "period", "possession_number"],
    how="left",
)
long_possession_movements_df["half_court_crossings"] = (
    long_possession_movements_df["half_court_crossings"].fillna(0).astype(int)
)
long_possession_movements_df = long_possession_movements_df[
    long_possession_movements_df["half_court_crossings"] < 2
].reset_index(drop=True)

print(f"Long possessions before half-court filter: {len(long_possessions_df)}")
print(f"Dropped all possessions crossing half court twice or more: {len(dropped_half_court_crossers_df)}")
print(f"Long possessions after half-court filter: {len(long_possessions_filtered_df)}")
display(long_possessions_filtered_df)

print(f"Tracking rows for filtered long possessions: {len(long_possession_movements_df)}")
display(long_possession_movements_df)
display(dropped_half_court_crossers_df)

if long_possession_movements_df.empty:
    print("No tracking rows matched the long possession summary rows.")
else:
    project_root = Path.cwd().parent if Path.cwd().name == "multi_game_data" else Path.cwd()
    if str(project_root) not in sys.path:
        sys.path.append(str(project_root))

    import funcs
    importlib.reload(funcs)

    funcs.plot_raw_movements_by_game_period(
        long_possession_movements_df,
        img_path="court.jpg",
        title="Long Possession Ball Movements (>24 Seconds)",
        line_alpha=0.75,
        line_width=1.5,
        marker_size=24,
    )


In [ ]:
# Full multi-game master dataframe with possessions crossing half court twice or more removed
def count_half_court_crossings(possession_df, half_court_x=47):
    possession_df = possession_df.sort_values("time", ascending=False)
    side = np.sign(possession_df["x"] - half_court_x).replace(0, np.nan).ffill().bfill()
    if side.empty or side.isna().all():
        return 0
    return int(side.ne(side.shift()).sum() - 1)

clean_half_court_crossings_df = (
    clean_possessions_df.groupby(["game_id", "period", "possession_number"])
    .apply(count_half_court_crossings)
    .rename("half_court_crossings")
    .reset_index()
)

clean_possessions_with_crossings_df = clean_possessions_df.merge(
    clean_half_court_crossings_df,
    on=["game_id", "period", "possession_number"],
    how="left",
)
clean_possessions_with_crossings_df["half_court_crossings"] = (
    clean_possessions_with_crossings_df["half_court_crossings"].fillna(0).astype(int)
)

dropped_clean_half_court_crossers_df = clean_possessions_with_crossings_df[
    clean_possessions_with_crossings_df["half_court_crossings"] >= 2
].copy()

clean_possessions_half_court_filtered_df = clean_possessions_with_crossings_df[
    clean_possessions_with_crossings_df["half_court_crossings"] < 2
].reset_index(drop=True)

print(f"Master tracking rows before half-court filter: {len(clean_possessions_df)}")
print(f"Master possessions before half-court filter: {clean_possessions_df[['game_id', 'period', 'possession_number']].drop_duplicates().shape[0]}")
print(f"Dropped possessions crossing half court twice or more: {dropped_clean_half_court_crossers_df[['game_id', 'period', 'possession_number']].drop_duplicates().shape[0]}")
print(f"Master tracking rows after half-court filter: {len(clean_possessions_half_court_filtered_df)}")
print(f"Master possessions after half-court filter: {clean_possessions_half_court_filtered_df[['game_id', 'period', 'possession_number']].drop_duplicates().shape[0]}")

display(clean_possessions_half_court_filtered_df)
display(dropped_clean_half_court_crossers_df)

clean_possessions_half_court_filtered_df.to_csv(
    "multi_game_data_half_court_filtered.csv",
    index=False,
)


In [ ]:
# Plot possessions removed by the full half-court crossing filter
if dropped_clean_half_court_crossers_df.empty:
    print("No possessions were removed by the half-court crossing filter.")
else:
    project_root = Path.cwd().parent if Path.cwd().name == "multi_game_data" else Path.cwd()
    if str(project_root) not in sys.path:
        sys.path.append(str(project_root))

    import funcs
    importlib.reload(funcs)

    removed_possessions_summary_df = (
        dropped_clean_half_court_crossers_df[
            ["game_id", "period", "possession_number", "half_court_crossings", "scored"]
        ]
        .drop_duplicates()
        .sort_values(["game_id", "period", "possession_number"])
        .reset_index(drop=True)
    )
    display(removed_possessions_summary_df)
    display(removed_possessions_summary_df["scored"].value_counts())

    funcs.plot_raw_movements_by_game_period(
        dropped_clean_half_court_crossers_df,
        img_path="court.jpg",
        title="Removed Possessions: Crossed Half Court Twice or More",
        line_alpha=0.75,
        line_width=1.5,
        marker_size=24,
    )


In [ ]:
clean_possessions_half_court_filtered_df

## Generating Threat Profiles for Multiple Games

Using the QUT and bootstrap methodology in `QUT_Bootstrap.ipynb`, we generate threat profiles for 5 more games chosen at random. These will help us construct a heuristic model in the future that can be used to test if we can reconstruct a simple threat profile using the data available to us.

In [ ]:
flipped = funcs.bin_and_flip(clean_possessions_half_court_filtered_df)
flipped


In [ ]:
flipped_filter = flipped[flipped["game_id"] != "0021500009"]
games = list(flipped_filter["game_id"].unique())
select = random.sample(games, 5)
select

In [ ]:
bootstrap_qut_results = {}
n_bootstraps = 500
mc_null = 500
alpha = 0.05

for game in select:
    bootstrap_qut_results[game] = funcs.analyze_tv_logistic_single_game(
        flipped_filter,
        game_id=game,
        game_col="game_id",
        n_bootstraps=n_bootstraps,
        n_possessions=1000,
        samples_per_possession=100,
        min_samples_required=100,
        mc_null=mc_null,
        alpha=alpha,
        court_image_path="court.jpg",
        show_progress=True,
        plot_beta_summary=True,
        plot_pointwise_maps=True,
        plot_lambda_distributions=True,
        seed=42,
    )

bootstrap_qut_summary_df = pd.DataFrame.from_dict(
    {game: result["summary"] for game, result in bootstrap_qut_results.items()},
    orient="index",
)
bootstrap_qut_summary_df.index.name = "game_id"
bootstrap_qut_summary_df
